In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.graph_objects as go


%config InlineBackend.figure_format='svg' 

from morphing_birds import Pigeon3D, plot_plotly, animate_plotly, animate



## Load Raw Data to Inspect Missing Markers

In [ ]:

csv_path = "/Users/lfrance/Library/CloudStorage/OneDrive-TheAlanTuringInstitute/002_Projects/Hawkflight/morphing_birds/data/2025-08-28-FullPigeons.csv"
df = pd.read_csv(csv_path)


In [8]:
# Pick a row with no nans
frame_idx = df.dropna().index[3]

coord_cols = [c for c in df.columns if c.endswith(('_x', '_y', '_z'))]
bases = sorted({c.rsplit('_', 1)[0] for c in coord_cols})

points, labels = [], []
for base in bases:
    x_col, y_col, z_col = f"{base}_x", f"{base}_y", f"{base}_z"
    if x_col in df and y_col in df and z_col in df:
        x, y, z = df.loc[frame_idx, [x_col, y_col, z_col]].astype(float)
        if np.isfinite([x, y, z]).all():
            points.append([x, y, z])
            labels.append(base)

points = np.array(points)
cx, cy, cz = points.mean(axis=0)
mins, maxs = points.min(axis=0), points.max(axis=0)
ranges = maxs - mins
d = ranges.max() / 2  # half-range for equal axes

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode="markers",  # no text drawn; hover only
            marker=dict(size=4),
            text=labels,
            hovertemplate="%{text}<extra></extra>"
        )
    ]
)

fig.update_layout(
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        xaxis=dict(range=[cx - d, cx + d]),
        yaxis=dict(range=[cy - d, cy + d]),
        zaxis=dict(range=[cz - d, cz + d]),
        aspectmode="cube"  # equal scale on all axes
    ),
    width=700, height=700,  # square figure
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

csv_path = "/Users/lfrance/Library/CloudStorage/OneDrive-TheAlanTuringInstitute/002_Projects/Hawkflight/morphing_birds/data/2025-08-28-FullPigeons.csv"
df = pd.read_csv(csv_path)

# Identify marker bases from columns that end with _x/_y/_z
coord_cols = [c for c in df.columns if c.endswith(('_x', '_y', '_z'))]
bases = sorted({c.rsplit('_', 1)[0] for c in coord_cols})

# Count frames where each marker has all 3 coords present and finite
counts = []
for base in bases:
    x, y, z = f"{base}_x", f"{base}_y", f"{base}_z"
    if {x, y, z}.issubset(df.columns):
        block = df[[x, y, z]]
        present = block.notna().all(axis=1) & np.isfinite(block.to_numpy()).all(axis=1)
        counts.append((base, int(present.sum())))

# Build summary table
summary = pd.DataFrame(counts, columns=["marker", "count"])
summary["total_frames"] = len(df)
summary["percent"] = summary["count"] / summary["total_frames"]
summary = summary.sort_values("count", ascending=True)

# Horizontal bar chart: unreliable (low count) at top
fig = go.Figure(go.Bar(
    x=summary["count"],
    y=summary["marker"],
    orientation="h",
    text=[f"{p:.1%}" for p in summary["percent"]],
    hovertemplate="Marker: %{y}<br>Non-NaN frames: %{x} / " + f"{len(df)}" + "<br>Percent: %{text}<extra></extra>"
))

fig.update_layout(
    title="Non-NaN frame count per marker",
    xaxis_title="Frames with valid (x,y,z)",
    yaxis_title="Marker",
    height=max(400, 18 * len(summary)),  # auto-grow with marker count
    margin=dict(l=140, r=10, t=40, b=40)
)

fig.show()

## Plotting Pigeon Shape

In [9]:
pigeon3d = Pigeon3D("../data/mean_pigeon_shape.csv")
plot_plotly(pigeon3d)


In get_marker_names_full:
Desired order length: 16
Available markers before filtering: 15
Fixed markers being removed: ['head', 'centre_backpack', 'centre_body_base', 'left_tailbase', 'right_tailbase']
Final canonical order length: 15
Markers in canonical order but not available: ['centre_body_base']
Markers available but not in canonical order: []

Initialized polygons:
head: [11, 15, 10]
body: [11, 19, 17, 18, 10]
tail: [19, 17, 18, 12, 13, 14]
right_armwing: [11, 9, 3, 5, 7, 19]
left_armwing: [10, 8, 2, 4, 6, 18]
left_handwing: [2, 4, 0]
right_handwing: [3, 5, 1]
Input keypoints shape: (1, 15, 3)
Number of markers in self.marker_names: 15
Number of markers in self.marker_index: 15
Marker names: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_lastsecondary_tip', 'right_lastsecondary_tip', 'left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder', 'left_tailtip', 'centre_tailtip', 'right_tailtip']
Marker indices: [0, 1, 2

In [13]:

# Create pigeon
pigeon = Pigeon3D("../data/mean_pigeon_shape.csv", use_simple=False)

# Load all data in one step  
motion_data, info_df, valid_frames = pigeon.load_data_complete(
    "../data/2025-08-28-FullPigeons.csv"
)

# Animate to any frame
# pigeon.update_to_frame(motion_data, 200)

# Plot
animate_plotly(pigeon, motion_data[0:200])


In get_marker_names_full:
Desired order length: 16
Available markers before filtering: 15
Fixed markers being removed: ['head', 'centre_backpack', 'centre_body_base', 'left_tailbase', 'right_tailbase']
Final canonical order length: 15
Markers in canonical order but not available: ['centre_body_base']
Markers available but not in canonical order: []

Initialized 7 polygon sections:
  head: 3 vertices
  body: 5 vertices
  tail: 6 vertices
  right_armwing: 6 vertices
  left_armwing: 6 vertices
  left_handwing: 3 vertices
  right_handwing: 3 vertices
Input keypoints shape: (1, 15, 3)
Number of markers in self.marker_names: 15
Number of markers in self.marker_index: 15
Marker names: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_lastsecondary_tip', 'right_lastsecondary_tip', 'left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder', 'left_tailtip', 'centre_tailtip', 'right_tailtip']
Marker indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 